# EIS Page Viewer

Simple notebook to view EIS document pages for manual review and exploration.

In [1]:
import re
import pandas as pd
from pathlib import Path

# Display settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

## Load Data

In [2]:
# Load CE documents
documents = pd.read_parquet('../data/analysis/documents_combined.parquet')
eis_documents = documents[documents["dataset_source"] == "EIS"]
#print(f"Loaded {len(eis_documents):,} EIS documents")

def page_num_int(value):
    """Extract first numeric token from page labels like '1', 'Page-1', or '1-6'."""
    if pd.isna(value):
        return pd.NA
    match = re.search(r"(\d+)", str(value))
    if match:
        return int(match.group(1))
    return pd.NA

# Load CE pages
EIS_pages = pd.read_parquet('../data/processed/EIS/pages.parquet')
EIS_pages["_page_number_num"] = EIS_pages["page_number"].map(page_num_int).astype("Int64")
print(f"Loaded {len(EIS_pages):,} EIS pages")

Loaded 6,131,757 EIS pages


## Helper Functions

In [3]:
def list_documents(project_id):
    """List all documents for a given project."""
    docs = eis_documents[eis_documents["project_id"] == project_id]
    if len(docs) == 0:
        print(f"No documents found for project_id: {project_id}")
        return None
    
    print(f"\nDocuments for project: {project_id}")
    print("-" * 80)
    display_cols = ["document_id", "document_type", "document_type_category", 
                    "main_document", "file_name", "total_pages"]
    available_cols = [c for c in display_cols if c in docs.columns]
    return docs[available_cols]

In [4]:
def view_page(document_id, page_num=1):
    """View a specific page from a document."""
    page_target = page_num_int(page_num)
    if pd.isna(page_target):
        print(f"Invalid page number: {page_num}")
        return None

    page = EIS_pages[(EIS_pages["document_id"] == document_id) & 
                    (EIS_pages["_page_number_num"] == page_target)]
    
    if len(page) == 0:
        print(f"Page {page_num} not found for document_id: {document_id}")
        return None
    
    # Get document metadata
    doc = eis_documents[eis_documents["document_id"] == document_id]
    if len(doc) > 0:
        print(f"\n{'='*80}")
        print(f"Document: {doc.iloc[0].get('file_name', 'N/A')}")
        print(f"Type: {doc.iloc[0].get('document_type', 'N/A')} | "
              f"Category: {doc.iloc[0].get('document_type_category', 'N/A')} | "
              f"Main: {doc.iloc[0].get('main_document', 'N/A')}")
        print(f"{'='*80}")
    
    print(f"\n--- PAGE {page_num} ---\n")
    print(page.iloc[0]["page_text"])
    print(f"\n--- END PAGE {page_num} ---\n")
    return page.iloc[0]["page_text"]

In [5]:
def view_pages(document_id, start_page=1, end_page=5):
    """View a range of pages from a document."""
    # Get document metadata
    doc = eis_documents[eis_documents["document_id"] == document_id]
    if len(doc) > 0:
        print(f"\n{'='*80}")
        print(f"Document: {doc.iloc[0].get('file_name', 'N/A')}")
        print(f"Type: {doc.iloc[0].get('document_type', 'N/A')} | "
              f"Category: {doc.iloc[0].get('document_type_category', 'N/A')} | "
              f"Main: {doc.iloc[0].get('main_document', 'N/A')}")
        print(f"{'='*80}")
    
    for page_num in range(start_page, end_page + 1):
        page = EIS_pages[(EIS_pages["document_id"] == document_id) & 
                        (EIS_pages["_page_number_num"] == page_num)]
        if len(page) > 0:
            print(f"\n--- PAGE {page_num} ---\n")
            print(page.iloc[0]["page_text"])
            print(f"\n--- END PAGE {page_num} ---\n")
        else:
            print(f"Page {page_num} not found")
            break

In [6]:
def view_project_pages(project_id, document_id=None):
    """View all pages of EISch document in a project (no truncation).
    
    Args:
        project_id: The project ID to filter documents
        document_id: Optional document ID to view only that specific document
    """
    docs = eis_documents[eis_documents["project_id"] == project_id]
    
    # Filter by document_id if provided
    if document_id is not None:
        docs = docs[docs["document_id"] == document_id]
        
    if len(docs) == 0:
        if document_id is not None:
            print(f"No document found with document_id: {document_id} in project_id: {project_id}")
        else:
            print(f"No documents found for project_id: {project_id}")
        return
        
    for _, doc in docs.iterrows():
        doc_id = doc["document_id"]
        print(f"\n{'='*80}")
        print(f"Document: {doc.get('file_name', 'N/A')}")
        print(
            f"Type: {doc.get('document_type', 'N/A')} | "
            f"Category: {doc.get('document_type_category', 'N/A')} | "
            f"Main: {doc.get('main_document', 'N/A')}"
        )
        print(f"{'='*80}")
                
        doc_pages = (
            EIS_pages[EIS_pages["document_id"] == doc_id]
            .sort_values(["_page_number_num", "page_number"], na_position="last")
        )
                
        for _, page in doc_pages.iterrows():
            print(f"\n--- PAGE {page['page_number']} ---\n")
            print(page["page_text"])

In [7]:
def search_pages(project_id, search_term):
    """Search for a term in all pages of a project."""
    docs = eis_documents[eis_documents["project_id"] == project_id]
    doc_ids = docs["document_id"].tolist()
    
    matching_pages = eis_pages[
        (eis_pages["document_id"].isin(doc_ids)) & 
        (eis_pages["page_text"].str.contains(search_term, case=False, na=False))
    ]
    
    print(f"\nFound {len(matching_pages)} pages containing '{search_term}'")
    print("-" * 80)
    
    for _, page in matching_pages.iterrows():
        doc = docs[docs["document_id"] == page["document_id"]].iloc[0]
        print(f"\nDocument: {doc.get('file_name', 'N/A')} | Page: {page['page_number']}")
        
        # Show context around the search term
        text = page["page_text"]
        idx = text.lower().find(search_term.lower())
        if idx >= 0:
            start = max(0, idx - 200)
            end = min(len(text), idx + len(search_term) + 200)
            context = text[start:end]
            if start > 0:
                context = "..." + context
            if end < len(text):
                context = context + "..."
            print(context)
    
    return matching_pages

In [8]:
def get_random_project(n_docs=2):
    """Get a random project with at lEISst n_docs documents."""
    import random
    doc_counts = eis_documents.groupby("project_id").size()
    eligible = doc_counts[doc_counts >= n_docs].index.tolist()
    
    project_id = random.choice(eligible)
    print(f"Random project with {doc_counts[project_id]} documents: {project_id}")
    return project_id

## Explore a specific project

Set the `project_id` below and run the cells to explore.

In [9]:
# Set your project ID here
#project_id = "d9c3d975f3e8c38c549f8182bec4181b" # 
#project_id = "9e2d0d5d3ae33f94a782b79bae9db894" # 
project_id = "5df42becc385c43ab0d1977d31d43e96" # 

In [10]:
# List all documents for this project
list_documents(project_id)


Documents for project: 5df42becc385c43ab0d1977d31d43e96
--------------------------------------------------------------------------------


,document_id,document_type,document_type_category,main_document,file_name,total_pages
45943,2d7fc15f66766c718b6c3a7d20404c48,,other,NO,1695.pdf,27
45944,afebca84bee5cb1317311a236a208908,,other,NO,Dear Interested Party Letter_BMM.Juniper.EIS_03.31.22 signed.pdf,2
45945,076b7caffdf6a6fc1f934eb0ce7b92d6,DEIS,draft,YES,Juniper_DEIS_Boards.pdf,5
45946,ca357dc43b5bd0f68151b1aacd3ccc0c,DEIS,draft,YES,1695.pdf,396
45947,71567784e7e4b4ae7333edae565a74c8,,other,NO,1695.pdf,3
45948,05763975e5a69597097cb0e6fd911883,,other,NO,Juniper-EIS_Scoping-Mtg_Presentation.pdf,23
45949,18625f77c03a44f5c181db25fd7ccb7a,,other,NO,Juniper_EIS_Scoping_Factsheet.pdf,4
45950,c1639b4ccd606083a5af293badb06b60,DEIS,draft,YES,Juniper_DEIS_Trifold_Boards.pdf,3
45951,b79a16e9ba503e7b489abbf48c2d2328,,other,NO,1695.pdf,11
45952,01c5b01c83f23865e302b3b038f1541d,,draft,NO,Juniper_DEIS_Mtg-Presentation.pdf,26


In [30]:
# View first 3 pages of EISch document
view_project_pages(project_id, document_id = "e3f5668ea8c43b988cf6385ad8996656")
#view_project_pages(project_id)


Document: 113.pdf
Type: DEIS | Category: draft | Main: YES

--- PAGE 1 ---

 
Document Type: 
EIS-Administrative Record 
 
Index Field: 
Draft EIS 
 
Project Name: 
North Alabama Utility-Scale 
Solar Facility 
 
Project Number: 
2020-06 
 
 
 
 
 
 
 
 
 
NORTH ALABAMA UTILITY-SCALE SOLAR FACILITY 
DRAFT ENVIRONMENTAL IMPACT STATEMENT 
Lawrence County, Alabama 
 
Prepared by: 
TENNESSEE VALLEY AUTHORITY 
Knoxville, Tennessee 
 
 
January 2021 
 


--- PAGE 2 ---

This page intentionally left blank 
 


--- PAGE 3 ---

 
 
Cover Sheet 
 
Draft Environmental Impact Statement 
i 
COVER SHEET 
North Alabama Utility-Scale Solar Facility EIS 
Proposed action: 
The Tennessee Valley Authority (TVA) prepared 
this environmental impact statement (EIS) to 
assess the potential environmental effects of the 
proposed North Alabama Utility-Scale Solar Facility 
in Lawrence County, Alabama. Under the Proposed 
Action, TVA would purchase the 2,896-acre Project 
Site two miles east of Courtland and co

## View Specific Document

In [ ]:
# Get document IDs for the project
docs = eis_documents[eis_documents["project_id"] == project_id]
docs[["document_id", "file_name", "main_document", "document_type"]]

,document_id,file_name,main_document,document_type
73227,65c6bb9f-ea78-d49b-3513-f5fc9c09da57,DOI-BLM-CO-N010-2021-0043-CX-DR_Road 1509 Emergency Repair_for web.pdf,NO,OTHER
73228,25334806-7f45-75fb-d4b0-cad8e1e5567a,DOI-BLM-CO-N010-2021-0043-CX_Road 1509 Emergency Repair_for web.pdf,YES,CE


In [ ]:
# Set document_id and view specific pages
document_id = docs.iloc[0]["document_id"]  # First document
view_pages(document_id, start_page=1, end_page=5)


Document: DOI-BLM-CO-N010-2021-0043-CX-DR_Road 1509 Emergency Repair_for web.pdf
Type: OTHER | Category: other | Main: NO
Page 1 not found


## SEISrch for Text

In [ ]:
# SEISrch for dates or specific text
sEISrch_pages(project_id, "22")


Found 1 pages containing '22'
--------------------------------------------------------------------------------

Document: cx-007803.pdf | Page: 1-4
...of any spilled material shall
begin immediately.

10. All potential pitfalls to wildlife will be covered or filled when not attended.

Limetto King acting for
Linda Hughes
NEPA Compliance Officer

11/22/2011
Date

5


document_id page_number  \
73021  d0da2533-5bbd-70b4-796f-d8d0145303fe         1-4   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [ ]:
# SEISrch for decision-related text
sEISrch_pages(project_id, "approved")


Found 2 pages containing 'approved'
--------------------------------------------------------------------------------

Document: DOI-BLM-CO-N010-2021-0043-CX-DR_Road 1509 Emergency Repair_for web.pdf | Page: 1-4
...e emergency repair of BLM Road 1509 in Moffat County, Colorado,
near the Little Snake/White River Field Office boundary. The repair is expected to consist of
layering compatible rock material and BLM-approved Geotech material, then installing a culvert
and armoring the inlet and outlet of the culvert with local rock material from a nearby rock pile,
within BLM standards. The road repairs are expected to b...

Document: DOI-BLM-CO-N010-2021-0043-CX_Road 1509 Emergency Repair_for web.pdf | Page: 1-11
...safety.
Conformance with the Land Use Plan
The Proposed Action is subject to and is in conformance (43 CFR 1610.5) with the following
land use plan:
Land Use Plan: Little Snake Record of Decision and Approved Resource Management Plan
(ROD/RMP)
Date Approved: October 2011
Decisi

document_id page_number  \
4688  65c6bb9f-ea78-d49b-3513-f5fc9c09da57         1-4   
4689  25334806-7f45-75fb-d4b0-cad8e1e5567a        1-11   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                